In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
import rpy2.situation
for row in rpy2.situation.iter_info():
    print(row)
from rpy2.robjects import r
from rpy2.robjects.packages import importr
%matplotlib qt
from local_functions import pix_to_deg,read_data,calculate_bf,plot_bfs,shrink_axes,make_subplots, inset_axes
from statsmodels.formula.api import probit
from statsmodels.formula.api import ols
from scipy.stats import linregress

import rpy2.robjects as ro
ro.r('.libPaths(c(.libPaths(), "/Library/Frameworks/R.framework/Versions/4.6/Resources/library"))')
bf_package=importr('BayesFactor')

# A4 dimensions in inches
a4_width = 8.27 
a4_height = 11.69 

task = 'exp3'

subj_colors = sns.color_palette("Spectral",21)
colors = sns.diverging_palette(250, 30, l=70, n=12,center='dark')



### read data

In [ ]:
all_epochs = []
bids_dir = '/Volumes/LaCie2/2021_Occlusion_behavioural/data'
subjids = [i for i in np.arange(1,21)]
all_epochs = read_data(bids_dir,subjids,task = 'exp3')

In [ ]:
gaze_error = []
beh_error = []

i = 0
condition = 'occluded'
epos = all_epochs[i].copy()
epos = epos[epos.metadata.condition==condition]
x_idx = np.where([ii=='xpos' for ii in epos.ch_names])[0][0]
epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

beh_error.append(epos.metadata.loc[(epos.metadata.condition=='occluded'),'rt'].to_numpy())


tv = epos.times.copy()
end_time = epos.metadata.loc[:,'occlusion_end_time']
end_time_idx = np.array([np.argmin(np.abs(tv-i)) for i in end_time])



gaze_error.append([epos._data[trial_i,x_idx,trial_end_idx] for trial_i,trial_end_idx in enumerate(end_time_idx)])




### make plot

In [ ]:
# pulling trialwise data from each participant, making a plot per person
plt.close('all')
temp_error_transparent_all = []
temp_error_occluded_all = []
error_deg_transparent_all = []
error_deg_occluded_all = []
final_gaze_error_occluded = []
final_gaze_error_transparent = []
trialwise_gaze_spatial_error = []
trialwise_behavioral_spatial_error = []

for i in range(20):
    epos = all_epochs[i].copy()
    fig,axs = plt.subplots(2,2,num=i,figsize=(16,8))
    temp_error_transparent_all.append(epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).rt-epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time)
    temp_error_occluded_all.append(epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).rt-epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time)
    
    ax = axs[0][0]
    ax.plot(temp_error_transparent_all[i],'grey',alpha=0.7,label='transparent')
    ax.plot(temp_error_occluded_all[i],'red',alpha=0.7,label='occluded')
    ax.set_xlabel('velocity')
    ax.set_ylabel('temporal error (s)')
    ax.set_ylim(-0.2,1.6)
    ax.legend()
    error_deg_occluded_all.append(temp_error_occluded_all[i].index*temp_error_occluded_all[i])
    error_deg_transparent_all.append(temp_error_transparent_all[i].index*temp_error_transparent_all[i])
    
    ax = axs[0][1]
    ax.plot(error_deg_transparent_all[i],'grey',alpha=0.7,label='transparent')
    ax.plot(error_deg_occluded_all[i],'red',alpha=0.7,label='occluded')
    ax.set_xlabel('velocity')
    ax.set_ylabel('spatial error (dva)')
    ax.set_ylim(-1.3,10)
    ax.legend()
    fig.suptitle(f'sub-S{str(i+1).zfill(2)}',weight='bold')
    # eye tracking
    for condition,ax in zip(['transparent', 'occluded'],[axs[1][0],axs[1][1]]):
        epos = all_epochs[i].copy()
        epos = epos[epos.metadata.condition==condition]
        x_idx = np.where([ii=='xpos' for ii in epos.ch_names])[0][0]
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1
        speeds = np.sort(epos.metadata.velocity.unique())
        for speed_idx in range(len(speeds)):
            end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv>end_time]=np.nan
            trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
            trials[:,tv>end_time]=np.nan
            [ax.plot(tv,trial,color=colors[speed_idx],lw=0.2,alpha=1) for trial in trials]
            ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
            ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=15,zorder=500,markeredgecolor='k')

        ax.set_ylabel('x-gaze position (dva)')
        ax.set_xlabel('time (s) since trial start')
        ax.set_title(condition)
        ax.set_ylim(-12,12)

        # trialwise spatial errors (save out for later)
        end_times = epos.metadata['occlusion_end_time'].values
        velocities = epos.metadata['velocity'].values
        gaze_at_end = np.array([epos._data[t, x_idx, np.argmin(np.abs(epos.times - end_times[t]))] for t in range(len(epos))])
        df = epos.metadata[['velocity','direction','rt','occlusion_end_time']].copy()
        df['gaze_spatial_error'] = np.abs(epos.metadata.occluder_start_pos_deg)-gaze_at_end
        df['behavioral_spatial_error'] = (epos.metadata['rt'].values - end_times) * velocities
        df['condition'] = condition
        df['subject'] = i + 1
        trialwise_gaze_spatial_error.append(df)

    plt.tight_layout()
    plt.savefig(f'/Volumes/LaCie2/2021_Occlusion_behavioural/data/exp3/derivatives/figures/individual_data/trialwise_errors/sub-S{str(i+1).zfill(2)}_trialwise_errors.png')
                

trialwise_df = pd.concat(trialwise_gaze_spatial_error, ignore_index=True)

In [ ]:

for i in range(20):
    plt.close('all')
    fig,axs = plt.subplots(6,2,num=i,figsize=(7,16),sharex=True,sharey=True)

    for condition in (['transparent','occluded']):
        epos = all_epochs[i].copy()
        epos = epos[epos.metadata.condition==condition]
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

        for ax,speed_idx in zip(axs.flatten(),range(len(speeds))):
            end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv>end_time]=np.nan
            trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
            trials[:,tv>end_time]=np.nan


            if condition =='transparent':
                ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=0.4,label='transparent')
            else:
                ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1,label='occluded')
            ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=8,zorder=500,markeredgecolor='k')
            ax.set_ylim(-12,12)

            ax.legend(fontsize=6,frameon=False)

    fig.supylabel('x-gaze position (dva)')
    fig.supxlabel('time (s) since trial start')
    fig.suptitle(f'sub-S{str(i+1).zfill(2)}',weight='bold')

    plt.tight_layout()

    plt.savefig(f'/Volumes/LaCie2/2021_Occlusion_behavioural/data/exp3/derivatives/figures/individual_data/sub-S{str(i+1).zfill(2)}_eyetracking_transparent-vs-occluded.png')

In [ ]:
plt.close('all')
fig,axs = plt.subplots(3,2,num=1,figsize=(a4_width, a4_height*0.9))


epos = all_epochs[0].copy()
x_pos = epos.metadata.occlusion_start_time.unique()[0]

# Labels for the subplots
labels = ['(a)', '(b)', '(c)','(d)','(e)','(f)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):
    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1 + 0.02 
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='right')


#************************************
### (A) TRIAL
#************************************
ax = shrink_axes(axs[0][0],0.95,0.8)

summary_dat = epos.metadata.loc[epos.metadata.direction==1,:].groupby('velocity').mean(numeric_only=True)
for i in range(len(summary_dat)):
    traj = np.linspace(summary_dat['start_pos_deg'].iloc[i],np.abs(summary_dat['start_pos_deg'].iloc[i]),int(summary_dat['n_frames'].iloc[i]))
    # traj[traj>np.abs(summary_dat['occluder_start_pos_deg'].iloc[i])]=np.nan
    traj_tv = np.linspace(0,summary_dat['total_time'].iloc[i],int(summary_dat['n_frames'].iloc[i]))


    ax.plot(traj_tv[(traj>=-4.5)&(traj<=4.5)],traj[(traj>=-4.5)&(traj<=4.5)],':',color=colors[i])
    ax.plot(traj_tv[(traj<-4.5)],traj[(traj<-4.5)],'-',color=colors[i],lw=1.5)



x_pos = 0.2
ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, 5),xytext=(x_pos, 10),ha='left', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)

for speed_idx in range(len(speeds)):
    ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)


ax.set_ylim([-13,13])
ax.set_xlabel('time since trial start (s)')
ax.set_ylabel('x-position object (dva)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


# #************************************
# ### (B) BEHAVIOUR
# #************************************
ax = shrink_axes(axs[0][1],0.85,0.8)

[ax.plot(error_deg_occluded_all[i].index,error_deg_occluded_all[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
ax.plot(error_deg_occluded_all[i].index,np.mean(error_deg_occluded_all,axis=0),'k',alpha=1,lw=2,label='occluded')
for speed_idx in range(len(speeds)):
    ax.plot(error_deg_occluded_all[0].index[speed_idx],np.mean(error_deg_occluded_all,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.plot(error_deg_transparent_all[i].index,np.mean(error_deg_transparent_all,axis=0),'k:',alpha=1,lw=2,label='non-occluded')
ax.legend(frameon=False,fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('error (dva)')



# #************************************
# ### (C) EYETRACKING -- individual Sub-07 
# #************************************
sample_p = 6
ax = shrink_axes(axs[1][0],1,1)
[ax.spines[loc].set_visible(False) for loc in ['top','bottom','right','left']]
ax.set_xticks([])
ax.set_yticks([])

ax.set_ylabel('x-gaze position (dva)',labelpad=15)
ax.set_xlabel('time since trial start (s)',labelpad=8)

bottoms = [0.55,0.025]
condition_labels = [f'(i) sub-S{str(sample_p+1).zfill(2)}: occluded',f'(ii) sub-S{str(sample_p+1).zfill(2)}: non-occluded']
for condition_i,condition in enumerate(['occluded','transparent']):
    ax_in = make_subplots(ax,bottom=bottoms[condition_i],height="45%")
    x_ind = np.where([i=='xpos' for i in all_epochs[0].info.ch_names])[0][0]

    epos = all_epochs[sample_p].copy()
    epos = epos[epos.metadata.condition==condition]
    x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
    epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

    speeds = np.sort(epos.metadata.velocity.unique())


    for speed_idx in range(len(speeds)):
        end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
        tv = epos.times.copy()
        tv[tv>end_time]=np.nan
        trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
        trials[:,tv>end_time]=np.nan

        [ax_in.plot(tv,trial,color=colors[speed_idx],lw=0.2,alpha=.5) for trial in trials]
        ax_in.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=0.8,alpha=1)

        ax_in.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)


    ax_in.spines['top'].set_visible(False)
    ax_in.spines['right'].set_visible(False)

    if condition_i == 0:
        ax_in.set_xticklabels('')
    ax_in.text(0.01, .98, condition_labels[condition_i], transform=ax_in.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='left')
    ax_in.set_ylim([-13,13])


# #************************************
# ### (D) EYETRACKING -- group
# #************************************
ax = shrink_axes(axs[1][1],1,1)
[ax.spines[loc].set_visible(False) for loc in ['top','bottom','right','left']]
ax.set_xticks([])
ax.set_yticks([])

ax.set_ylabel('x-gaze position (dva)',labelpad=15)
ax.set_xlabel('time since trial start (s)',labelpad=8)

bottoms = [0.55,0.025]
condition_labels = [f'(i) Average (n=20): occluded',f'(ii) Average (n=20): non-occluded']

for condition_i, condition in enumerate(['occluded', 'transparent']):
    ax_in = make_subplots(ax, bottom=bottoms[condition_i], height="45%")
    x_ind = np.where([i == 'xpos' for i in all_epochs[0].info.ch_names])[0][0]

    # correct trajectories (like in a)
    for i in range(len(summary_dat)):
        traj = np.linspace(summary_dat['start_pos_deg'].iloc[i],np.abs(summary_dat['start_pos_deg'].iloc[i]),int(summary_dat['n_frames'].iloc[i]))
        traj_tv = np.linspace(0,summary_dat['total_time'].iloc[i],int(summary_dat['n_frames'].iloc[i]))
        ax_in.plot(traj_tv[(traj>=-4.5)&(traj<=4.5)],traj[(traj>=-4.5)&(traj<=4.5)],':',color=colors[i],lw=0.5)
        ax_in.plot(traj_tv[(traj<-4.5)],traj[(traj<-4.5)],'-',color=colors[i],lw=0.5)

    # eyetracking results
    speeds = np.sort(all_epochs[0].metadata.velocity.unique())  # get speeds from first participant

    for speed_idx in range(len(speeds)):
        all_participant_means = []

        for epos in all_epochs:  # loop over participants instead of sample_p
            epos = epos.copy()
            epos = epos[epos.metadata.condition == condition]
            x_idx = np.where([i == 'xpos' for i in epos.ch_names])[0][0]
            epos._data[epos.metadata.direction == 2, x_idx, :] = epos._data[epos.metadata.direction == 2, x_idx, :] * -1

            end_time = epos.metadata.loc[epos.metadata.velocity == speeds[speed_idx], 'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv > end_time] = np.nan
            trials = epos._data[epos.metadata.velocity == speeds[speed_idx], x_idx, :]
            trials[:, tv > end_time] = np.nan

            all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean

        grand_mean = np.nanmean(all_participant_means, axis=0)  # average across participants

        # plot individual participant means as faint lines (replaces single-trial lines)
        # [ax_in.plot(tv, p_mean, color=colors[speed_idx], lw=0.2, alpha=.5) for p_mean in all_participant_means]
        ax_in.plot(tv, grand_mean, color=colors[speed_idx], lw=0.8, alpha=1)
        ax_in.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)





    ax_in.spines['top'].set_visible(False)
    ax_in.spines['right'].set_visible(False)
    if condition_i == 0:
        ax_in.set_xticklabels('')
    ax_in.text(0.01, .98, condition_labels[condition_i], transform=ax_in.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='left')
    ax_in.set_ylim([-13, 13])


# #************************************
# ### (E) & (F) CORRELATIONS 
# #************************************
condition_labels = [' occluded',' non-occluded']
for condition_i,condition in enumerate(['occluded','transparent']):
    corrs = trialwise_df.loc[trialwise_df.condition==condition,:].groupby('subject').apply(lambda x: x['gaze_spatial_error'].corr(x['behavioral_spatial_error']))

    ax = shrink_axes(axs[2][condition_i],.9,.95)
    for s in range(1,21):
        x = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s),'gaze_spatial_error']
        y = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s),'behavioral_spatial_error']

        ax.plot(x,y,'.',color=subj_colors[s],alpha=0.4,markeredgecolor=None)

    
    ax.set_xlabel('error gaze position at occlusion end (dva)')
    if condition_i==0:
        ax.set_ylabel('error behavior (dva)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'{condition_labels[condition_i]}:\n r = {np.mean(corrs):.2f}',loc='left',y=0.8,fontsize=9)

    ax.set_xlim([-5,15])
    ax.set_ylim([-5,40])

    ### inset for distributions of correlations
    axins = inset_axes(ax, width="30%", height="35%", loc="upper right")

    vplot=axins.violinplot(corrs, showmeans=True)

    for pc in vplot['bodies']:
        pc.set_facecolor('black')
        pc.set_edgecolor('black')
    for partname in ('cmeans', 'cmaxes', 'cmins', 'cbars'):
        if partname in vplot:
            vplot[partname].set_color('black')
            vplot[partname].set_linewidth(1.5)


    [axins.plot((s/75-10/75)+1,corrs.to_numpy()[s],'.',color=subj_colors[s],alpha=0.6) for s in range(20)]

    # axins.yaxis.tick_right()
    axins.yaxis.set_label_position("right")
    axins.set_xticks([])
    axins.set_ylabel('Correlation',fontsize=7)
    axins.set_ylim([-0.7,0.7])

    axins.spines['top'].set_visible(False)
    axins.spines['right'].set_visible(False)


    bf_out = calculate_bf(np.expand_dims(corrs,axis=1))
    print(f'mean correlation behaviour and gaze error for condition {condition} = {np.mean(corrs)}, BF = 10^{np.log10(np.array(bf_out))}')




# #************************************
# ### SAVE
# #************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3.pdf')


In [ ]:
# Correlation per speed

fig,axs = plt.subplots(6,2,figsize=(6,13))

condition_labels = [' occluded',' non-occluded']
for ax_curr,vel in zip(axs.flatten(), np.sort(trialwise_df.velocity.unique())):
    condition_i = 0
    condition = 'occluded'
    corrs = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.velocity==vel),:].groupby('subject').apply(lambda x: x['gaze_spatial_error'].corr(x['behavioral_spatial_error']))


    ax = shrink_axes(ax_curr,.9,.95)
    for s in range(1,21):
        x = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s)&(trialwise_df.velocity==vel),'gaze_spatial_error']
        y = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s)&(trialwise_df.velocity==vel),'behavioral_spatial_error']

        ax.plot(x,y,'.',color=subj_colors[s],alpha=0.4,markeredgecolor=None)

    

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'{condition_labels[condition_i]}, velocity {str(np.round(vel,2))} deg/sec:\n r = {np.mean(corrs):.2f}',loc='left',y=0.8,fontsize=9)

    ax.set_xlim([-5,15])
    ax.set_ylim([-5,40])

fig.supxlabel('error gaze position at occlusion end (dva)')
fig.supylabel('error behavior (dva)')
plt.tight_layout()


plt.savefig(f'/Volumes/LaCie2/2021_Occlusion_behavioural/data/exp3/derivatives/figures/correlation_per_speed.png')
plt.savefig(f'/Volumes/LaCie2/2021_Occlusion_behavioural/data/exp3/derivatives/figures/correlation_per_speed.pdf')

